# Predicting the SG Trend Index with the Paper's Momentum Layer

The paper fits its shared momentum network to MM positions. Here we keep the same architecture but use **daily SG Trend Index returns as the training target**, so the positions it learns are the energy positions that best explain the largest CTAs' P&L. We compare them with a simple benchmark and with the MM-trained NN.

- **Fitted directly to daily SG returns, the paper's momentum layer explains them out of sample** (daily correlation 0.33, 2015–2025), but not better than the fixed 20/120 rule (0.36).
- **It puts more weight on long horizons than the MM network**, with its largest weight at 250 days, the longest horizon available. The MM network concentrates on 30–45 days.

This is one of three SG notebooks, meant to be read in order:

1. [`nn_benchmark.ipynb`](nn_benchmark.ipynb): does the NN's trend component earn CTA-like P&L? SG returns are used only as a **benchmark**.
2. [`predict_sg_index.ipynb`](predict_sg_index.ipynb): the paper's momentum layer with SG returns as the **training target**.
3. [`sg_input_feature.ipynb`](sg_input_feature.ipynb): SG-derived **input features** as corrections to the MM nowcast.

Helpers are in [`src/sg_cta.py`](src/sg_cta.py).

## Setup

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'working_ideas' else Path.cwd()
PROJECT_ROOT = REPO_ROOT / 'paper_replication'
sys.path[:0] = [str(PROJECT_ROOT / 'src'), str(REPO_ROOT / 'working_ideas' / 'src')]

import torch
from paper_replication.config import HORIZONS, ReplicationConfig
import sg_cta as sg
from sg_cta import KEYS, MARKETS, PNL_LABELS

OUTPUT = PROJECT_ROOT / 'outputs' / 'predict_sg_index'
OUTPUT.mkdir(parents=True, exist_ok=True)
torch.set_num_threads(1)
config = ReplicationConfig(data_dir=PROJECT_ROOT / 'data_required', basis='futures_only',
                           train_weeks=104, seed=7)
COLORS = {'trend': '#277f8e', 'bias': '#b7782c', 'rule': '#96587e', 'sg': '#596eaf'}
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False})

## Data

We build the same daily momentum inputs and weekly panel as the replication, and reuse the MM-trained NN's per-window horizon loadings from `run_replication` for comparison.

In [ ]:
panel, daily = sg.load_weekly_data(config)
daily = sg.add_trend_rule(daily)
sg_levels = sg.load_sg_log_levels(config.data_dir)
_, mm_loadings = sg.run_decomposition(panel, config)

## Fitting the paper's momentum layer to SG returns

The [benchmark notebook](nn_benchmark.ipynb) used positions learned from MM. Here we learn positions from SG returns instead, keeping the paper's architecture: 50 momentum horizons, three shared trend factors and the reaction function $R(u)=u\,e^{(1-u^2)/2}$. We fit it to **daily** SG Trend Index returns:

$$r^{SG}_d = c + \sum_{m}h_\theta\big(x_{m,d-1}\big)\,\frac{\Delta P_{m,d}}{\nu_{m,d}/\sqrt{252}}+\varepsilon_d,\qquad h_\theta(x)=\sum_{k=1}^{3}v_k\,R\Big(\sum_n w_k(n)\,x(n)\Big).$$

$h_\theta$ is the risk-scaled position CTAs hold going into day $d$, set from the previous close; $\Delta P/\nu$ is the day's volatility-scaled price change on the held contract. Two choices differ from the MM network:

- **One set of trend weights $v$ for all six markets.** SG returns cannot say how exposure is split inside the oil complex (the paper's objection), so we do not ask them to.
- **Daily data.** The P&L identity holds every day, which gives about five times the observations of weekly data. Energy P&L lines up with SG returns on the same day, with no lead or lag.

**Out-of-sample design.** For each year from 2015 to 2025 we train on the previous four years and predict that year. The L1 penalty on $w$ is chosen from four values using the last training year as validation, so the test year never influences it. The 20/120 rule, with a scale fitted on the same windows, is the benchmark.

In [ ]:
daily_data = sg.daily_sg_arrays(daily, sg_levels, 'SG Trend')
print(f"{len(daily_data['dates']):,} trading days, {daily_data['dates'][0]:%Y-%m-%d} to {daily_data['dates'][-1]:%Y-%m-%d}")
sg_run = sg.rolling_sg_fit(daily_data, config)
print('Chosen L1 penalty by year:', sg_run['penalty'])

fits = {'SG-fitted momentum network': sg_run['fitted'], '20/120 rule (benchmark)': sg_run['rule_fitted']}
display(pd.DataFrame({name: sg.fit_scores(f, sg_run['actual']) for name, f in fits.items()}).T.round(3))
display(sg.yearly_correlations(fits, sg_run['actual']).round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for values, label, color in [(mm_loadings, 'NN fitted to MM positions', COLORS['trend']),
                             (sg.sg_horizon_loadings(sg_run['models']), 'Network fitted to SG returns', COLORS['sg'])]:
    profile = values.mean(0)
    ax.plot(HORIZONS, profile / np.abs(profile).sum(), marker='o', ms=3, color=color, label=label)
ax.axhline(0, color='black', lw=.8)
ax.set(title='Linearized weight on each momentum horizon (average over fits, scaled to unit absolute sum)',
       xlabel='Momentum horizon (trading days)', ylabel='Relative weight')
ax.grid(alpha=.25)
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT / 'horizon_profiles.png', dpi=160)
plt.show()

**Reading.** Positions learned only from SG returns explain SG out of sample in every year, strongly in 2015–2022 and weakly in 2023–2025, matching the benchmark notebook. Flexibility does not beat the fixed 20/120 rule, so four years of daily data are not enough to learn a better trend rule from an index in which energy is a small slice.

The horizon profiles differ:
- **MM network:** concentrated on 30–45-day momentum.
- **SG-fitted network:** noisier and spread over longer horizons, with its largest weight at the longest horizon available (250 days). CTAs' energy P&L looks slower than the MM flow the paper models, consistent with Benhamou et al. (2026), who find about a third of SG Trend exposure at a 500-day horizon.

Both profiles put negative weight on 5–10-day momentum.

## Save results

In [ ]:
pd.DataFrame({'sg_fitted': sg_run['fitted'], 'rule_fitted': sg_run['rule_fitted'],
              'sg_actual': sg_run['actual']}).to_csv(OUTPUT / 'sg_daily_fit.csv')
pd.Series(sg_run['penalty'], name='l1_penalty').to_csv(OUTPUT / 'chosen_penalty.csv')

## References

- Sun, Bouchouev and Fattouh (2026), [Momentum Trading and Managed Money Positioning in Energy](https://www.oxfordenergy.org/wpcms/wp-content/uploads/2026/03/Insight-177-Money-Positioning-in-Energy.pdf), OIES Insight 177, §§1 and 3.
- Société Générale Prime Services, [SG Trend Index methodology](https://wholesale.banking.societegenerale.com/fileadmin/indices_feeds/SG_Trend_Index_Methodology.pdf) and [SG Trend Indicator methodology](https://wholesale.banking.societegenerale.com/fileadmin/indices_feeds/SG_Trend_Indicator_Methodology_Summary.pdf).
- CFTC, [Disaggregated COT explanatory notes](https://www.cftc.gov/MarketReports/CommitmentsofTraders/DisaggregatedExplanatoryNotes/index.htm): definition of Managed Money.
- Boos and Grob (2023), [Tracking speculative trading](https://www.sciencedirect.com/science/article/pii/S1386418122000635), *Journal of Financial Markets* 64.
- Roncalli and Teiletche (2008), [An Alternative Approach to Alternative Beta](https://doi.org/10.2139/ssrn.1035521): Kalman-filter replication of hedge fund indices.
- Benhamou, Ohana, Guez and Jacquot (2026), [Decoding CTA Allocations by Trend Horizon](https://rpc.cfainstitute.org/blogs/enterprising-investor/2026/decoding-cta-allocations-by-trend-horizon), CFA Institute.